In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf # Deep learning framework used to build, train, and evaluate neural networks
from tensorflow.keras.preprocessing.image import ImageDataGenerator # Library for real-time image augmentation and rescale operations during training
from tensorflow.keras.applications import MobileNetV2 # Pre-trained light-weight convolutional neural network model used for transfer learning
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D # Core layer types: Global pooling to reduce spatial dimensions, Dense layer for classification
from tensorflow.keras.models import Model # Model API used to group layers into an object with training and inference features
from tensorflow.keras.optimizers import Adam # Optimization algorithm used to update network weights iteratively based on training data
import os # Operating system interface, useful for handling directory paths and file system navigation
from tensorflow.keras.models import load_model # Function used to load saved model structures and pre-trained weights
from tensorflow.keras.preprocessing import image # Module providing utilities for loading and preprocessing raw images for prediction
import numpy as np # Library for numerical operations, used to work with multi-dimensional image arrays
from tensorflow.keras.callbacks import EarlyStopping # Callback to stop training when a monitored metric has stopped improving, preventing overfitting

In [ ]:
img_size=(224, 224)  # standard input resolution expected by MobileNetV2 to capture optimal features
batch_size=32        # Number of samples processed before the model's internal parameters are updated

In [ ]:
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = datagen.flow_from_directory(

    '/content/drive/MyDrive/food  classification/Food Classification dataset',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

Found 10329 images belonging to 34 classes.


In [ ]:
val_data = datagen.flow_from_directory(
     '/content/drive/MyDrive/food  classification/Food Classification dataset',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 2572 images belonging to 34 classes.


In [ ]:
# Load the pre-trained MobileNetV2 model architecture with ImageNet weights
base = MobileNetV2(
    include_top=False,  # Excludes the final fully-connected classification layer so we can attach a custom classifier
    weights='imagenet',  # Uses feature extraction patterns learned from the large-scale ImageNet dataset
    input_shape=(224, 224, 3)  # Defines the expected input shape as 224x224 pixels with 3 color channels (RGB)
)

base.trainable = False  # Freezes the base feature extractor layers to preserve pre-trained patterns during initial setup

# Reduce spatial dimensions of feature maps to a single vector per channel, preventing parameter explosion
x = GlobalAveragePooling2D()(base.output)

# Add a fully connected hidden layer to learn specialized features for this specific task
x = Dense(256, activation='relu')(x)

# Final classification layer with 4 output units corresponding to our 4 categories (using softmax for class probabilities)
output = Dense(34, activation='softmax')(x)

# Define the complete computational graph by linking inputs and outputs
model = Model(
    inputs=base.input,
    outputs=output
)

model.summary()  # Displays the architecture layout, layer connections, and trainable/non-trainable parameter count

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,594,658 (9.90 MB)

 Trainable params: 336,674 (1.28 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
base.trainable = True  # Unfreezes the MobileNetV2 layers to prepare for fine-tuning

# Calculate the midpoint to divide the network layers for partial freezing
midpoint = len(base.layers) // 2  # Computes the index dividing the base model layers into two halves

# Keep the early layers frozen as they capture low-level, generic features (edges, textures) that transfer well
for layer in base.layers[:midpoint]:
    layer.trainable = False

# Unfreeze late layers to let them adapt specifically to the unique visual patterns of brain tumor classification
for layer in base.layers[midpoint:]:
    layer.trainable = True

# Print the status of each layer in MobileNetV2 to verify that layers are frozen or unfrozen correctly
for layer in base.layers:
    print(layer.name, layer.trainable)

input_layer False
Conv1 False
bn_Conv1 False
Conv1_relu False
expanded_conv_depthwise False
expanded_conv_depthwise_BN False
expanded_conv_depthwise_relu False
expanded_conv_project False
expanded_conv_project_BN False
block_1_expand False
block_1_expand_BN False
block_1_expand_relu False
block_1_pad False
block_1_depthwise False
block_1_depthwise_BN False
block_1_depthwise_relu False
block_1_project False
block_1_project_BN False
block_2_expand False
block_2_expand_BN False
block_2_expand_relu False
block_2_depthwise False
block_2_depthwise_BN False
block_2_depthwise_relu False
block_2_project False
block_2_project_BN False
block_2_add False
block_3_expand False
block_3_expand_BN False
block_3_expand_relu False
block_3_pad False
block_3_depthwise False
block_3_depthwise_BN False
block_3_depthwise_relu False
block_3_project False
block_3_project_BN False
block_4_expand False
block_4_expand_BN False
block_4_expand_relu False
block_4_depthwise False
block_4_depthwise_BN False
block_4_dep

In [9]:
#Set up early stopping to prevent overfitting by monitoring validation loss and stopping if performance plateaus
early_stop = EarlyStopping(
    patience=3,
    restore_best_weights=True
)

# Configure training parameters: learning rate is set low (0.0001) for fine-tuning to avoid destroying pre-trained weights
model.compile(
    optimizer=Adam(0.0001),
    loss='categorical_crossentropy', # Loss function for multi-class classification tasks with one-hot encoded labels
    metrics=['accuracy'] # Metric to trace training and validation precision
)

# Train the model with the configured generators and parameters
history = model.fit(
    train_data,  # Provides augmented batches of training data
    epochs=5,  # Number of complete runs through the full dataset
    validation_data=val_data, # Validation dataset used to evaluate generalization performance at each epoch
    callbacks=[early_stop] # Uncomment when using early stopping callback
)

Epoch 1/5
188/323 ━━━━━━━━━━━━━━━━━━━━ 39:08 17s/step - accuracy: 0.7199 - loss: 0.9978

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


323/323 ━━━━━━━━━━━━━━━━━━━━ 7279s 22s/step - accuracy: 0.8241 - loss: 0.6074 - val_accuracy: 0.8328 - val_loss: 0.5535
Epoch 2/5
323/323 ━━━━━━━━━━━━━━━━━━━━ 1099s 3s/step - accuracy: 0.9439 - loss: 0.1960 - val_accuracy: 0.8736 - val_loss: 0.4423
Epoch 3/5
323/323 ━━━━━━━━━━━━━━━━━━━━ 1114s 3s/step - accuracy: 0.9747 - loss: 0.0867 - val_accuracy: 0.8705 - val_loss: 0.4879
Epoch 4/5
323/323 ━━━━━━━━━━━━━━━━━━━━ 1117s 3s/step - accuracy: 0.9863 - loss: 0.0506 - val_accuracy: 0.8678 - val_loss: 0.5244
Epoch 5/5
323/323 ━━━━━━━━━━━━━━━━━━━━ 1078s 3s/step - accuracy: 0.9932 - loss: 0.0284 - val_accuracy: 0.8530 - val_loss: 0.6045


In [10]:
# Load the target test image and resize it to match the network's input requirements
img = image.load_img('/content/drive/MyDrive/food  classification/Food Classification dataset/Donut/Donut (1).jpeg', target_size=(224,224))

# Convert the loaded PIL image structure into a numpy numerical array for computation
img_array = image.img_to_array(img)

# Normalize pixel values to the [0, 1] range to align with training preprocessing steps
img_array = img_array / 255.0

# Add a batch dimension at axis 0 because Keras models accept inputs in batches (e.g., (1, 224, 224, 3))
img_array = np.expand_dims(img_array, axis=0)

# Generate predictions for the target image, yielding probability values for each class
prediction = model.predict(img_array)

# Identify the index containing the highest predicted probability
predicted_class = np.argmax(prediction)

# Retrieve the sorted list of class names mapping to the prediction indices from the training generator
class_names = list(train_data.class_indices.keys())

# Output the predicted class name and corresponding probability confidence score
print("Predicted class:", class_names[predicted_class])
print("Probability:", prediction[0][predicted_class])

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Predicted class: Donut
Probability: 0.99446267
